# 🔬 Notebook 3: Key-Value Store — Deep Dive: Consistent hashing, Quorums, Anti-entropy

## 🛠️ Setup

```bash
cd 06-system-designs/key-value-store
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1 — consistent hashing in ~30 lines

In [ ]:
import hashlib, bisect
from collections import defaultdict

def h(s: str) -> int:
    return int(hashlib.md5(s.encode()).hexdigest(), 16)

class Ring:
    def __init__(self, vnodes_per_node=64):
        self.vnodes_per_node = vnodes_per_node
        self.ring: list[tuple[int, str]] = []    # (position, node)
        self.sorted_positions: list[int] = []

    def add_node(self, node: str):
        for i in range(self.vnodes_per_node):
            pos = h(f"{node}#{i}")
            self.ring.append((pos, node))
        self.ring.sort()
        self.sorted_positions = [p for p,_ in self.ring]

    def owner(self, key: str) -> str:
        pos = h(key)
        i = bisect.bisect_right(self.sorted_positions, pos) % len(self.ring)
        return self.ring[i][1]

    def replicas(self, key: str, n: int) -> list[str]:
        pos = h(key)
        i = bisect.bisect_right(self.sorted_positions, pos) % len(self.ring)
        out: list[str] = []
        seen = set()
        while len(out) < n:
            node = self.ring[i % len(self.ring)][1]
            if node not in seen:
                seen.add(node); out.append(node)
            i += 1
            if i > len(self.ring) + n * 10:
                break  # safety
        return out

r = Ring()
for n in ["A","B","C","D"]:
    r.add_node(n)

# Check load balance
owners = defaultdict(int)
for k in range(10_000):
    owners[r.owner(f"key-{k}")] += 1
print("Key distribution:", dict(owners))

# What happens when we add a new node?
r2 = Ring()
for n in ["A","B","C","D","E"]:
    r2.add_node(n)

moved = sum(1 for k in range(10_000) if r.owner(f"key-{k}") != r2.owner(f"key-{k}"))
print(f"Adding node E moves {moved}/10000 keys "
      f"(~{moved/100:.1f}% — close to 1/5 = 20%, the theoretical fair share)")


## Deep dive 2 — a tiny quorum simulator

In [ ]:
import random

class Replica:
    def __init__(self, name, fail_prob=0.0):
        self.name = name
        self.store: dict[str, tuple[str,int]] = {}   # key -> (value, version)
        self.fail_prob = fail_prob

    def write(self, key, value, version):
        if random.random() < self.fail_prob: return False
        cur = self.store.get(key)
        if cur is None or version > cur[1]:
            self.store[key] = (value, version)
        return True

    def read(self, key):
        if random.random() < self.fail_prob: return None
        return self.store.get(key)

class Coordinator:
    def __init__(self, replicas, N, W, R):
        self.replicas = replicas; self.N=N; self.W=W; self.R=R
        self.version = 0

    def put(self, key, value):
        self.version += 1
        acks = 0
        for r in self.replicas:
            if r.write(key, value, self.version):
                acks += 1
                if acks >= self.W:
                    return {"status": "ok", "acks": acks}
        return {"status": "fail", "acks": acks}

    def get(self, key):
        responses = []
        for r in self.replicas:
            v = r.read(key)
            if v is not None:
                responses.append(v)
                if len(responses) >= self.R:
                    break
        if not responses: return None
        # pick the highest version
        return max(responses, key=lambda x: x[1])

random.seed(0)
replicas = [Replica(f"r{i}", fail_prob=0.2) for i in range(3)]
c = Coordinator(replicas, N=3, W=2, R=2)

print(c.put("user:42", "alice"))
print(c.put("user:42", "alice-updated"))
print("Read:", c.get("user:42"))

# If we set W=3 with 20% failure rate, writes will sometimes fail
c2 = Coordinator(replicas, N=3, W=3, R=1)
results = [c2.put(f"k{i}", "v") for i in range(50)]
fails = sum(1 for r in results if r["status"] == "fail")
print(f"\nWith W=3, {fails}/50 writes failed due to replica timeouts")


## Deep dive 3 — anti-entropy with Merkle trees

Two replicas drift apart (some writes went to one but not the other). How do we *find* the
differing keys efficiently without comparing every key?

**Merkle tree**: build a tree where each leaf is hash(key, value) and each internal node is
hash of its children. Two replicas compare roots; if different, they recurse down **only**
into differing branches. Dramatic bandwidth savings: O(log N) for small diffs.

```
        H(root)
        /     \
     H(L)     H(R)      ← compare roots; differ → recurse
     /  \     /  \
   ...  ...  ...  ...    ← only descend into differing subtrees
```

Cassandra uses this for repair; DynamoDB uses a similar anti-entropy mechanism.
